[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jsonmen/bias-and-variance/blob/main/understand-ml/linear-regression/LinearRegression.ipynb)

## 🧠 Intuitive Linear Regression Explanation

**Imagine this:**

You sell lemonade. Every day, you write down:

* the outside temperature ☀️
* how many cups of lemonade you sold 🥤

For example:

| Temperature (°C) | Cups of Lemonade |
| ---------------- | ---------------- |
| 20               | 10               |
| 25               | 20               |
| 30               | 30               |

Now you want to predict: **how many cups of lemonade you will sell when it’s 35°C outside.**

Linear regression can help you with that. This algorithm finds the **best-fit line** through your data. Once you have that line, you can use it to predict lemonade sales for any given temperature.

The prediction formula looks like this:

$$
y = w \cdot x + b
$$

In plain words:
`cups of lemonade = some weight * temperature + some bias`.

This method isn't perfectly accurate. Its error (or inaccuracy) can be measured using the **Mean Squared Error (MSE)**:

$$
\text{MSE} = \frac{1}{N} \sum_{i=1}^{N} (y_i - \hat{y}_i)^2
$$

Or in simpler terms:
`error = mean of (true cups of lemonade - predicted cups)^2`.

To compute `weight` and `bias`, we use **[OLS](#📘-OLS-(Ordinary-Least-Squares))** — the Ordinary Least Squares method.

> Of course, you can also use **gradient descent** to get $w$ and $b$, and it's a great way to *learn* about optimization. But in the real world, people rarely use gradient descent for simple linear regression — OLS is faster and gives an exact solution.

![lemonade_regression_plot.png|700](https://i.imgur.com/EkNbM6d.png)

## 📘 OLS (Ordinary Least Squares)

In linear regression, OLS gives us the optimal parameters for the best-fit line. Also need to mention that OLS only works well when features are not highly collinear and the number of samples is larger than the number of features.

Here’s the formula:

$$
\beta = (X^T X)^{-1} X^T y
$$

Let’s break it down:

* **$X$** is your input matrix of shape $(n_{\text{samples}}, n_{\text{features}})$. If you want to include the bias term $b$, you need to add a column of ones to $X$.
* **$y$** is your target vector with shape $(n_{\text{samples}}, )$.
* **$\beta$** is the parameter vector. It contains the **bias** (intercept) at the first position and the **weights** for each feature after that. Also need to mention if $X$ doesn't full rank matrix then $\beta$ doesn't have solutions.

### Now, term by term:

* **$X^T X$**
  This is the **feature covariance matrix** — it tells you how much each feature correlates with the others.

* **$(X^T X)^{-1}$**
  The **inverse** of the covariance matrix. It adjusts for redundancy and relationships between features, revealing the **unique contribution** of each one.

* **$X^T y$**
  This is like a "raw importance" vector — it tells you how much each feature correlates with the target. It doesn’t consider how features relate to each other, though.

* **Putting it all together**:
  $\beta = (X^T X)^{-1} X^T y$
  This equation solves a system of linear equations while minimizing the squared error between predictions and actual values.

### 🧠 Bonus: What are we minimizing?

The goal of linear regression is to minimize this cost function:

$$
\min_\beta \; J(\beta) = \|X \beta - y\|^2
$$

This is the **sum of squared residuals** — we want our predictions to be as close as possible to the real values.

### Preparing

In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

In [2]:
X = np.array([[15], [23], [31], [27], [35]])
Y = np.array([20, 28, 30, 25, 32])
print(f"{X.shape=}, {Y.shape=}")

X.shape=(5, 1), Y.shape=(5,)


# OLS Linear Regression

In [3]:
def lr_forward(X, weigts, bias):
    return (X @ weigts) + bias

In [4]:
def ols(X, Y):
    assert len(X.shape)==2
    n_samples = X.shape[0]
    X_with_intercept = np.hstack([np.ones((n_samples, 1)), X])

    XTX = X_with_intercept.T @ X_with_intercept
    XTX_inv = np.linalg.inv(XTX)
    XTy = X_with_intercept.T @ Y
    coefs = XTX_inv @ XTy
    weights = coefs[1:]
    bias = coefs[0]
    return weights, bias

In [5]:
%%time
weights, bias = ols(X, Y)
diy_ols_error = mean_squared_error(Y, lr_forward(X, weights, bias))
print("Model error:", diy_ols_error)
print(f"Prameters: weights={weights.tolist()} and bias={bias.item()}\n")

Model error: 2.8837837837837803
Prameters: weights=[0.5574324324324332] and bias=12.395270270270236

CPU times: user 772 μs, sys: 0 ns, total: 772 μs
Wall time: 777 μs


In [6]:
%%time
sklearn_lr = LinearRegression().fit(X, Y)
sklearn_lr_error = mean_squared_error(Y, sklearn_lr.predict(X))
print("Model error:", sklearn_lr_error)
print(f"Prameters: weights={sklearn_lr.coef_.tolist()} and bias={sklearn_lr.intercept_.item()}\n")

Model error: 2.8837837837837794
Prameters: weights=[0.5574324324324326] and bias=12.395270270270267

CPU times: user 4 ms, sys: 2.81 ms, total: 6.8 ms
Wall time: 7.85 ms


As you can see, the results are absolutely identical — except for the speed, which I believe is due to scikit-learn’s additional layers of abstraction. Still, the fact that the results match those of a state-of-the-art Python package strongly suggests that the implementation is correct.